Python 3.11.9

In [ ]:
#!pip install pandas numpy matplotlib scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

**Cargo el fichero en formato excel:**

In [ ]:
#!pip install openpyxl

In [ ]:
#tengo el fichero en la misma carpeta
original_data = pd.read_excel('_RIRS_litotricia_ID_estudio_original_6MAYO.xlsx')
patient_data = original_data.copy()
patient_data.head()

In [ ]:
N_pat, N_columns = patient_data.shape
print(f"Number of patients: {N_pat}")
print(f"Number of columns: {N_columns}")
patient_data.dtypes

# Primeros ajustes de formato

Cambio de nombres de columnas con error en codificacion:

In [ ]:
patient_data.columns.values.tolist()

In [ ]:
patient_data.rename(columns={"radiografÃƒÂ\xada": "radiografia"}, inplace=True)

In [ ]:
patient_data.columns.values.tolist()[30:]

Quito decimal sobrante de variables categoricas:

In [ ]:
# quitar decimal sobrante de variables categoricas
categorias = ['Genero','Obesidad', 'Diabetes', 'HTA','Dislipemia', 'Fumador_tabaco', 'ASA',
              'ANOMALIAANATÓMICA','TIPOANOMALIA','LOCAL_COD','LOCALIZACIÓN','GRUPO',
              'cultivo','Cultresul','Microorganismo', 'GUY_SCORE',
              'CATETERJJPREVIO', 'Dilatacion_ureteral', 'Vaina_acceso', 'Calibre_vaina',
              'DERIVACIONPOSTERIOR', 'PULS',
              'MOTIVO', 'tipotto', 'lado','Complicaciones','tipocomplicacion','complicaciones_intra',
              'complicaciones_post','trasfusion','reintervcomplic','embolizacion','SANGRADO','PERFORACION',
              'INFECCION','SEPSIS','HEMATURIA','HEMATOMA', 'IngresoSiNo',
              'urgencias','REingreso','RESUELTOEN6m']	#'Compli_Post_Recod','Complic_Intra_Recodif'
patient_data[categorias] = patient_data[categorias].astype('Int64')
patient_data.head()

Redondear el procentaje de neutrofilos:

In [ ]:
patient_data['ProcentajeNeutrofilos'] = (patient_data['NeutrófilosTotales'] / patient_data['LeucocitosTotales']) * 100

In [ ]:
patient_data['ProcentajeNeutrofilos'] = patient_data['ProcentajeNeutrofilos'].round(decimals = 2)
patient_data.head()

# Identificación de errores y su eliminación

**Elección de las variables de interes**

Elegir solo las variables que se usan para un modelo completo (antes de nefrolitotomia y despues)

In [ ]:
patient_data.columns

In [ ]:
# Crear un nuevo DataFrame con columnas específicas
columnas_caract_antes = ['codigo', 'DIAGNÓSTICO', 'Genero', 'lado',
       'cultivo', 'Cultresul',
       'Microorganismo', 'altura', 'peso', 'BMI',  'Diabetes',
       'HTA', 'Dislipemia', 'Fumador_tabaco', 'ASA', 'ANOMALIAANATÓMICA',
       'TIPOANOMALIA', 'tipotto', 'NÚMEROLITIASIS', 'TAMAÑOLITIASIS',
       'VOLUMEN', 'UH', 'LOCAL_COD', 'LOCALIZACIÓN', 'GRUPO', 'GUY_SCORE','EDAD'] #'Obesidad',
columnas_quir=['CATETERJJPREVIO', 'Dilatacion_ureteral', 'Vaina_acceso',
       'Calibre_vaina', 'DERIVACIONPOSTERIOR', 'PULS']
columnas_analitica=['Procalcitonina', 'LeucocitosTotales', 'ProcentajeNeutrofilos'] #+['ProteinaCReactiva']
columnas_complicaciones = ['Complicaciones','tipocomplicacion','complicaciones_intra',
              'complicaciones_post','trasfusion','complicaclavien','reintervcomplic','embolizacion','SANGRADO','PERFORACION',
              'INFECCION','SEPSIS','HEMATURIA','HEMATOMA']+['IngresoSiNo'] #'PERDIDAACCESO',
etiqueta_a_predecir = ['RESUELTOEN6m']
## para_matriz = ['ComplInfSeps','ComplHemo']
#SELECCION DE COLUMNAS
selected_columns = columnas_caract_antes+columnas_quir+columnas_analitica+etiqueta_a_predecir+columnas_complicaciones
patient_data = patient_data[selected_columns]
patient_data.head()

### Arreglo preliminar de unas variables

**Umbralización de variables analíticas:**

In [ ]:
patient_data['Procalcitonina'] = np.where(
    patient_data['Procalcitonina'].notna(),
    np.where(patient_data['Procalcitonina'] >= 0.5, 1, 0),
    np.nan
)

patient_data['LeucocitosTotales'] = np.where(
    patient_data['LeucocitosTotales'].notna(),
    (
        (patient_data['LeucocitosTotales'] >= 15) |
        (patient_data['LeucocitosTotales'] <= 4.5)
    ).astype(int),
    np.nan
)

patient_data['ProcentajeNeutrofilos'] = np.where(
    patient_data['ProcentajeNeutrofilos'].notna(),
    (patient_data['ProcentajeNeutrofilos'] > 80).astype(int),
    np.nan
)

**calcular BMI donde es nan pero se dispone de altura y peso**

In [ ]:
import numpy as np

# guardar cuántos BMI faltaban antes
bmi_missing_before = patient_data['BMI'].isna().sum()

# condición:
# BMI vacío + altura disponible + peso disponible
mask = (
    patient_data['BMI'].isna() &
    patient_data['altura'].notna() &
    patient_data['peso'].notna()
)

# calcular BMI
patient_data.loc[mask, 'BMI'] = (
    patient_data.loc[mask, 'peso'] /
    ((patient_data.loc[mask, 'altura'] / 100) ** 2)
)

# opcional: redondear
patient_data['BMI'] = patient_data['BMI'].round(2)

# cuántos valores se recuperaron
bmi_missing_after = patient_data['BMI'].isna().sum()
recuperados = bmi_missing_before - bmi_missing_after

print(f"Valores de BMI recuperados: {recuperados}")

### Prospección de valores anómalos en variables númericas (aunque en este caso serían de interes)

In [ ]:
numericas = ['BMI','NÚMEROLITIASIS', 'TAMAÑOLITIASIS','VOLUMEN', 'UH','EDAD'] #+['ProteinaCReactiva']
# binarias
# catgeroicas nominales
# categoricas ordinales

In [ ]:
features = numericas
fig, axes = plt.subplots(nrows=len(features), ncols=3, figsize=(12, 3 * len(features)))
fig.suptitle("Distribuciones de caracteristicas numericas", fontsize=16)

plot_types = [
    ("Boxplot", lambda ax, data: ax.boxplot(data)),
    ("Histogram", lambda ax, data: ax.hist(data, bins=20, edgecolor='black')),
    ("Violin Plot", lambda ax, data: ax.violinplot(data))
]

for row_idx, feature in enumerate(features):
    column = patient_data[feature].dropna().values  # Convertimos en array 1D sin NaNs
    for col_idx, (title, plot_func) in enumerate(plot_types):
        ax = axes[row_idx, col_idx]
        plot_func(ax, column)  # Llamamos a la función de ploteo
        if col_idx == 0:  
            ax.set_ylabel(feature)  # Etiquetamos la fila con la variable
        ax.set_title(title)  # Ponemos el tipo de gráfico como título

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

Corregir BMI raro

In [ ]:
# máscara: BMI > 60 y con altura/peso disponibles
mask = (
    (patient_data['BMI'] > 60) &
    patient_data['altura'].notna() &
    patient_data['peso'].notna()
)

# cuántos valores se van a recalcular
n_recalculados = mask.sum()

# eliminar BMI previo
patient_data.loc[mask, 'BMI'] = pd.NA

# recalcular BMI
patient_data.loc[mask, 'BMI'] = (
    patient_data.loc[mask, 'peso'] /
    ((patient_data.loc[mask, 'altura'] / 100) ** 2)
)

# redondear
patient_data['BMI'] = patient_data['BMI'].round(2)

print(f"BMI recalculados: {n_recalculados}")

# segunda limpieza: cualquier BMI que siga siendo > 60 se invalida
mask_invalidos = patient_data['BMI'] > 60
n_eliminados = mask_invalidos.sum()

patient_data.loc[mask_invalidos, 'BMI'] = pd.NA

print(f"BMI eliminados por seguir >60: {n_eliminados}")

Volumen

In [ ]:
print(patient_data['VOLUMEN'].isna().sum())
patient_data = patient_data.drop(columns=['VOLUMEN'])
numericas.remove('VOLUMEN')
numericas

Acortar pacientes pedaitricos:

In [ ]:
len(patient_data)

In [ ]:
# Filtrar pacientes menores de 18
menores_18 = patient_data[patient_data["EDAD"] < 18]

# Agrupar y contar cuántas muestras hay por cada edad
desglose_edades = menores_18["EDAD"].value_counts().sort_index()

print("Desglose de muestras por edad (menores de 18):")
print(desglose_edades)

In [ ]:
patient_data[patient_data["EDAD"] < 18].sum

In [ ]:
# Eliminar pacientes menores de 18 años
patient_data = patient_data[patient_data["EDAD"] >= 18].reset_index(drop=True)
N_pat, N_columns = patient_data.shape
print(f"Number of patients: {N_pat}")

In [ ]:
features = numericas

fig, axes = plt.subplots(nrows=len(features), ncols=2, figsize=(10, 3 * len(features)))
fig.suptitle("Distribuciones de caracteristicas numericas", fontsize=16)

plot_types = [
    ("Boxplot", lambda ax, data: ax.boxplot(data)),
    ("Histogram", lambda ax, data: ax.hist(data, bins=20, edgecolor='black'))
]

for row_idx, feature in enumerate(features):
    column = patient_data[feature].dropna().values
    for col_idx, (title, plot_func) in enumerate(plot_types):
        ax = axes[row_idx, col_idx]
        plot_func(ax, column)
        if col_idx == 0:
            ax.set_ylabel(feature)
        ax.set_title(title)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
patient_data.loc[patient_data['NÚMEROLITIASIS'] == 0, 'NÚMEROLITIASIS'] = np.nan

In [ ]:
patient_data['NÚMEROLITIASIS'].unique()

In [ ]:
# filtrar muestras con BMI > 50
df_bmi50 = patient_data[patient_data['BMI'] > 50]
# ver todas las columnas (o las que te interesen)
df_bmi50

In [ ]:
patient_data[patient_data['BMI'] < 15]

In [ ]:
patient_data.columns

In [ ]:
patient_data.loc[patient_data['NÚMEROLITIASIS'] > 25, ['NÚMEROLITIASIS', 'TAMAÑOLITIASIS', 'GUY_SCORE','GRUPO']]

In [ ]:
patient_data.loc[patient_data['TAMAÑOLITIASIS'] > 40, ['NÚMEROLITIASIS', 'TAMAÑOLITIASIS', 'GUY_SCORE','GRUPO']]

In [ ]:
patient_data = patient_data.drop(columns=['altura','peso'])

#### identificación de errores en variables categoricas

In [ ]:
categoricas = [col for col in patient_data.columns if col not in numericas and col not in ['codigo','RESUELTOEN6m','ComplInfSeps','ComplHemo'] and col not in columnas_complicaciones]
categoricas

In [ ]:
#mirar frecuencias de los valores
# mirar valores diferentes del numero, como el fichero originalmente es de spss

In [ ]:
import matplotlib.pyplot as plt
for col in categoricas:
    print(f"\n--- {col} ---")
    
    # frecuencias en %
    freq = patient_data[col].value_counts(dropna=False, normalize=True) * 100
    print(freq.round(2))

    # pie plot
    plt.figure()
    freq.plot.pie(autopct='%1.1f%%', startangle=90)
    plt.title(f"Distribución de {col}")
    plt.ylabel("")  # quita etiqueta del eje
    plt.show()

In [ ]:
patient_data = patient_data.drop(columns=['DIAGNÓSTICO'])
categoricas.remove('DIAGNÓSTICO')

In [ ]:
mask_invalidos = patient_data['GRUPO'] == 7
n_eliminados = mask_invalidos.sum()
patient_data.loc[mask_invalidos, 'GRUPO'] = pd.NA
print(f"GRUPO = 7: {n_eliminados}")

In [ ]:
mask_invalidos = patient_data['LOCALIZACIÓN'] > 4
n_eliminados = mask_invalidos.sum()
patient_data.loc[mask_invalidos, 'LOCALIZACIÓN'] = pd.NA
print(f"LOCALIZACIÓN == 5,6,7,8: {n_eliminados}")

In [ ]:
#!pip install seaborn

# Arreglo de <N/A>

In [ ]:
patient_data.head()

In [ ]:
patient_data.iloc[:,20:]

In [ ]:
patient_data = patient_data.replace({pd.NA: np.nan})

In [ ]:
# todo el DataFrame
patient_data = patient_data.astype('object')
patient_data = patient_data.applymap(lambda x: np.nan if pd.isna(x) else x)

In [ ]:
patient_data.head()

# EDA

In [ ]:
patient_data[numericas].describe().T

In [ ]:
import seaborn as sns

# Calcular la matriz de correlación solo con variables numéricas
correlacion = patient_data[numericas].corr()
# Visualizar
plt.figure(figsize=(12, 8))
sns.heatmap(correlacion, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlación')
plt.tight_layout()
plt.show()

In [ ]:

mapeos = {
    'RESUELTOEN6m':         {0: 'No', 1: 'Sí'},
    'Genero':               {0: 'Hombre', 1: 'Mujer'},
    'Obesidad':             {0: 'No', 1: 'Sí'},
    'Diabetes':             {0: 'No', 1: 'DM insulina', 2: 'DM sin insulina'},
    'HTA':                  {0: 'No', 1: 'Sí'},
    'Dislipemia':           {0: 'No', 1: 'Sí'},
    'Fumador_tabaco':       {0: 'No', 1: 'Sí', 2:'Exfumador'},
    'lado':                 {0: 'Derecho', 1: 'Izquierdo', 2:'Bilateral', 3:'Trasplante'},
    'CATETERJJPREVIO':      {0: 'No', 1: 'Sí'},
    'Dilatacion_ureteral':  {0: 'No', 1: 'Sí'},
    'DERIVACIONPOSTERIOR':  {0: 'No', 1: 'Mono J', 2:'Doble J'},
    'cultivo':              {0: 'No', 1: 'Sí'},
    'Cultresul':            {0: 'Negativo', 1: 'Positivo', 2:'Contaminado'},
    'Microorganismo':       {0: 'Negativo', 1: 'Ureasa negativo', 2:'Ureasa positivo'},
    'ANOMALIAANATÓMICA':    {0: 'No', 1: 'Sí'},
    'TIPOANOMALIA':         {1: 'Divertículo calicial', 2: 'Estenosis ureteral', 3: 'EUPU', 4: 'Riñón en herradura', 5: 'Duplicidad ureteral incompleta', 6: 'Malrotación renal', 7: 'Estenosis de uretra', 8: 'Ectopia renal', 9: 'Duplicidad ureteral completa', 10: 'Trasplante renal'},
    'tipotto':              {0: 'Primario', 1: 'Post-ESWL', 2: 'Post-URS', 3: 'Post-NLP'},
    'GRUPO':                {1: '<2-1,5cm', 2: '>=2-1,5cm'},
    'LOCAL_COD':            {0: 'ureteral', 1: 'renal', 2:'ambas'},
    'LOCALIZACIÓN':         {1: 'Pelvis renal', 2: 'GCS', 3:'GCM', 4:'GCI'},
    'Calibre_vaina':        {0: '9-11', 1: '10-12'},
    'Procalcitonina':       {0: 'Rango normal', 1: 'Fuera del rango'},
    'LeucocitosTotales':    {0: 'Rango normal', 1: 'Fuera del rango'},
    'ProcentajeNeutrofilos':{0: 'Rango normal', 1: 'Fuera del rango'},
    #'ASA':                  {1: 'I', 2: 'II', 3: 'III', 4: 'IV'},
    #'GUY_SCORE':            {1: 'I', 2: 'II', 3: 'III', 4: 'IV'},
    'Vaina_acceso':         {0: 'No', 1: 'Sí'},
    #'PULS':                 {0: 'No', 1: 'Sí'},
}

In [ ]:
import pandas as pd

# ignorando NaNs igual que lo hace la SPSS
categoricas = [
    col for col in patient_data.columns
    if col not in numericas and col not in ['codigo']
]

# Construir tabla resumen en formato compacto con moda
filas = []

for col in categoricas:
    abs_freq = patient_data[col].value_counts(dropna=True)
    rel_freq = patient_data[col].value_counts(dropna=True, normalize=True) * 100
    
    resumen = pd.DataFrame({'n': abs_freq, '%': rel_freq.round(1)})
    
    # Calcular moda (puede haber más de una)
    moda_vals = patient_data[col].mode().tolist()
    
    # Aplicar mapeo si existe
    if col in mapeos:
        resumen.index = resumen.index.map(lambda x: mapeos[col].get(x, x))
        moda_vals = [mapeos[col].get(v, v) for v in moda_vals]
    
    moda_str = " / ".join(str(v) for v in moda_vals)
    
    # Construir string de frecuencias en formato "Valor: X%"
    freq_str = "  ".join(f"{val}: {row['%']}%" for val, row in resumen.iterrows())
    
    filas.append({
        'Variable': col,
        'Frequency per Possible Value': freq_str,
        'Mode': moda_str
    })

# Crear DataFrame final
tabla_final = pd.DataFrame(filas, columns=['Variable', 'Frequency per Possible Value', 'Mode'])

# Mostrar sin índice, con máximo ancho de columna
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
print(tabla_final.to_string(index=False))

In [ ]:
filas = []

for col in categoricas:
    abs_freq = patient_data[col].value_counts(dropna=False)
    rel_freq = patient_data[col].value_counts(dropna=False, normalize=True) * 100

    for valor, n in abs_freq.items():
        etiqueta = mapeos[col].get(valor, valor) if col in mapeos else valor
        filas.append({
            'Variable': col,
            'Categoría': etiqueta,
            'n': n,
            '%': round(rel_freq[valor], 2)
        })

tabla = pd.DataFrame(filas)
tabla

In [ ]:
# ignorando NaNs igual que lo hace la SPSS
categoricas = [
    col for col in patient_data.columns
    if col not in numericas and col not in ['codigo']
]
for col in categoricas:
    print(f"\n--- {col} ---")
    
    # Conteo absoluto y porcentaje ignorando NaNs
    abs_freq = patient_data[col].value_counts(dropna=True)
    rel_freq = patient_data[col].value_counts(dropna=True, normalize=True) * 100
    
    resumen = pd.DataFrame({'n': abs_freq, '%': rel_freq.round(2)})
    
    # Aplicar mapeo si existe
    if col in mapeos:
        resumen.index = resumen.index.map(lambda x: mapeos[col].get(x, x))
    
    print(resumen)

In [ ]:
import matplotlib.pyplot as plt

n_cols = 2
n_rows = (len(categoricas) + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(14, 3 * n_rows)
)

axes = axes.flatten()

for ax, col in zip(axes, categoricas):

    data = patient_data[col].dropna()

    if col in mapeos:
        data = data.map(mapeos[col]).dropna()

    counts = data.value_counts()

    if counts.empty:
        ax.set_visible(False)
        continue

    total = counts.sum()
    percentages = (counts / total * 100).round(1)

    wedges, _ = ax.pie(
        counts.values,
        labels=None,
        startangle=90
    )

    legend_labels = [
        f"{cat} ({pct}%)"
        for cat, pct in zip(counts.index.astype(str), percentages)
    ]

    ax.legend(
        wedges,
        legend_labels,
        loc="center left",
        bbox_to_anchor=(1, 0.5),
        fontsize=10,
        frameon=False,
        #prop={'weight': 'bold'}
    )

    ax.set_title(col, fontsize=12, fontweight='bold')

# eliminar ejes vacíos
for i in range(len(categoricas), len(axes)):
    fig.delaxes(axes[i])

plt.subplots_adjust(
    left=0.05,
    right=0.88,
    top=0.95,
    bottom=0.05,
    wspace=-0.3,
    hspace=0.2
)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Mapeo 1 -> Sí, 0 -> No
serie = patient_data["RESUELTOEN6m"].map({1: "Sí", 0: "No"})

# Ignorar NaN (listwise deletion estilo SPSS)
serie = serie.dropna()

# Conteo
counts = serie.value_counts()

# Pie chart
plt.figure(figsize=(6,6))
plt.pie(
    counts,
    labels=counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Éxito")

# Leyenda centrada con fondo blanco semitransparente
plt.legend(
    title="Éxito",
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    frameon=True,
    facecolor="white",
    framealpha=0.7
)

plt.show()

## p-valor

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt

feat_cat = 'RESUELTOEN6m'

n_cols = 3
n_rows = -(-len(numericas) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()

for idx, feature in enumerate(numericas):
    subset = patient_data[[feat_cat, feature]].dropna()

    num_cat0 = subset.loc[subset[feat_cat] == 0, feature]
    num_cat1 = subset.loc[subset[feat_cat] == 1, feature]

    if num_cat0.empty or num_cat1.empty:
        axes[idx].set_visible(False)
        continue

    axes[idx].boxplot([num_cat0, num_cat1], labels=['No Éxito', 'Éxito'])
    axes[idx].set_title(f"Distribución de {feature}")

    stat, p_value = stats.mannwhitneyu(num_cat0, num_cat1, alternative='two-sided')
    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
    axes[idx].set_xlabel(f"Mann-Whitney p={p_value:.4f} {sig}")

for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout(h_pad=2)
plt.show()

In [ ]:
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import math

# =========================================================
# CONFIG
# =========================================================
patient_data = patient_data.loc[:, ~patient_data.columns.duplicated()]

feat_cat1 = 'RESUELTOEN6m'

vars_separadas = ['TIPOANOMALIA']

results = []

# =========================================================
# FUNCIÓN PVALOR
# =========================================================
def format_pvalue(p):
    if p < 0.0001:
        return "<0.0001"
    return f"{p:.4f}"

# =========================================================
# FUNCIÓN VALIDACIÓN
# =========================================================
def variable_valida(df, yvar, xvar):

    if yvar == xvar:
        return False

    if xvar not in df.columns:
        return False

    subset = df[[yvar, xvar]].dropna()

    if subset.empty:
        return False

    if subset[xvar].nunique() < 2:
        return False

    tabla = pd.crosstab(
        subset[yvar],
        subset[xvar]
    )

    if tabla.shape[0] < 2 or tabla.shape[1] < 2:
        return False

    return True

# =========================================================
# FUNCIÓN HEATMAP
# =========================================================
def plot_contingencias(lista_variables):

    valid_vars = [
        v for v in lista_variables
        if variable_valida(patient_data, feat_cat1, v)
    ]

    if len(valid_vars) == 0:
        return

    n_cols = 2
    n_rows = math.ceil(len(valid_vars) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(14, n_rows * 4)
    )

    # Si solo hay 1 fila
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    axes = axes.flatten()

    for i, var in enumerate(valid_vars):

        subset = patient_data[[feat_cat1, var]].dropna().copy()

        # Mapeos
        subset[feat_cat1] = subset[feat_cat1].map(
            mapeos.get(feat_cat1, lambda x: str(x))
        )

        subset[var] = subset[var].map(
            mapeos.get(var, lambda x: str(x))
        )

        # Tablas
        raw_crosstab = pd.crosstab(
            subset[feat_cat1],
            subset[var]
        )

        norm_crosstab = pd.crosstab(
            subset[feat_cat1],
            subset[var],
            normalize='index'
        ) * 100

        # Chi-cuadrado
        _, pvalue, _, _ = chi2_contingency(raw_crosstab)

        # Heatmap
        sns.heatmap(
            norm_crosstab,
            ax=axes[i],
            annot=raw_crosstab,
            fmt='d',
            cmap='Blues',
            linewidths=0.5,
            cbar=False,
            vmin=0,
            vmax=100
        )

        axes[i].set_title(
            f"{var}\np={format_pvalue(pvalue)}",
            fontsize=10
        )

        axes[i].set_xlabel('')
        axes[i].set_ylabel(feat_cat1)

        # Rotación etiquetas
        axes[i].tick_params(axis='x')

        # Guardar resultados
        results.append({
            'Variable': var,
            'p-valor_num': pvalue,
            'p-valor': format_pvalue(pvalue)
        })

    # Ocultar ejes sobrantes
    for j in range(len(valid_vars), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

# =========================================================
# VARIABLES PRINCIPALES
# =========================================================
categoricas_main = [
    c for c in categoricas
    if c not in vars_separadas
    and c not in columnas_complicaciones
]

# =========================================================
# FIGURA PRINCIPAL
# =========================================================
plot_contingencias(
    categoricas_main
)

# =========================================================
# FIGURA SECUNDARIA
# =========================================================
plot_contingencias(
    vars_separadas
)

# =========================================================
# RESULTADOS
# =========================================================
results_df = pd.DataFrame(results)

results_df = (
    results_df #.sort_values(by='p-valor_num')
    .drop(columns='p-valor_num')
    .reset_index(drop=True)
)

results_df

## Matriz de correlacion

In [ ]:
import numpy as np

patient_data['ComplInfSeps'] = np.where(
    patient_data[['INFECCION', 'SEPSIS']].isna().any(axis=1),
    np.nan,
    ((patient_data['INFECCION'] == 1) | (patient_data['SEPSIS'] == 1)).astype(int)
)# Contar cuántas veces aparece el valor 1 en las columnas específicas
columnas = ["INFECCION", "SEPSIS", "ComplInfSeps"]
conteos = patient_data[columnas].apply(lambda col: (col == 1).sum())
# Mostrar los resultados
print(conteos)

In [ ]:
import numpy as np

cols = ['HEMATURIA', 'HEMATOMA', 'SANGRADO', 'trasfusion', 'embolizacion']

patient_data['ComplHemo'] = np.where(
    patient_data[cols].isna().any(axis=1),
    np.nan,
    (
        (patient_data['HEMATURIA'] == 1) |
        (patient_data['HEMATOMA'] == 1) |
        (patient_data['SANGRADO'] == 1) |
        (patient_data['trasfusion'] == 1) |
        (patient_data['embolizacion'] == 1)
    ).astype(int)
)
# Contar cuántas veces aparece el valor 1 en las columnas específicas
columnas = ['HEMATURIA','HEMATOMA','SANGRADO','trasfusion','embolizacion','ComplHemo']
conteos = patient_data[columnas].apply(lambda col: (col == 1).sum())
# Mostrar los resultados
print(conteos)

In [ ]:
columnas_complicaciones.remove('IngresoSiNo')
patient_data.drop(columns=columnas_complicaciones, inplace=True)

##### matriz de correlacion OLD

In [ ]:
import seaborn as sns
# Calcular la matriz de correlación
sacar_correlacion = patient_data.drop(columns=['codigo'])
correlacion = sacar_correlacion.corr()
# Visualizar la matriz de correlación con un heatmap
plt.figure(figsize=(20, 10))
sns.heatmap(correlacion, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlación')
plt.show()

In [ ]:
# Ver varianza de cada columna (0 = valor constante)
print(patient_data.var(numeric_only=True).sort_values())
# Ver % de NaNs por columna
print((patient_data.isnull().mean() * 100).sort_values(ascending=False).round(2))

## Complicaciones

In [ ]:
patient_data.columns

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Mapeo 1 -> Sí, 0 -> No
serie = patient_data["IngresoSiNo"].map({1: "Sí", 0: "No"})

# Ignorar NaN (listwise deletion estilo SPSS)
serie = serie.dropna()

# Conteo
counts = serie.value_counts()

# Pie chart
plt.figure(figsize=(6,6))
plt.pie(
    counts,
    labels=counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Necesidad de ingreso")

# Leyenda centrada con fondo blanco semitransparente
plt.legend(
    title="Necesidad de ingreso",
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    frameon=True,
    facecolor="white",
    framealpha=0.7
)

plt.show()

In [ ]:
patient_data["IngresoSiNo"].value_counts()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Mapeo 1 -> Sí, 0 -> No
serie = patient_data["ComplInfSeps"].map({1: "Sí", 0: "No"})

# Ignorar NaN (listwise deletion estilo SPSS)
serie = serie.dropna()

# Conteo
counts = serie.value_counts()

# Pie chart
plt.figure(figsize=(6,6))
plt.pie(
    counts,
    labels=counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Complicaciones de origen infeccioso")

# Leyenda centrada con fondo blanco semitransparente
plt.legend(
    title="Complicaciones de origen infeccioso",
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    frameon=True,
    facecolor="white",
    framealpha=0.7
)

plt.show()

In [ ]:
counts

In [ ]:
# Mapeo 1 -> Sí, 0 -> No
serie = patient_data["ComplHemo"].map({1: "Sí", 0: "No"})

# Ignorar NaN (listwise deletion estilo SPSS)
serie = serie.dropna()

# Conteo
counts = serie.value_counts()

# Pie chart
plt.figure(figsize=(6,6))
plt.pie(
    counts,
    labels=counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Complicaciones de origen hemorrágica")

# Leyenda centrada con fondo blanco semitransparente
plt.legend(
    title="Complicaciones de origen hemorrágica",
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    frameon=True,
    facecolor="white",
    framealpha=0.7
)

plt.show()

In [ ]:
counts

### p-valores

In [ ]:
import numpy as np
np.random.seed(42)

ingreso

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd

feat_cat = 'IngresoSiNo'

n_cols = 3
n_rows = -(-len(numericas) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()

resultados = []

for idx, feature in enumerate(numericas):
    subset = patient_data[[feat_cat, feature]].dropna()

    num_cat0 = subset.loc[subset[feat_cat] == 0, feature]
    num_cat1 = subset.loc[subset[feat_cat] == 1, feature]

    if num_cat0.empty or num_cat1.empty:
        axes[idx].set_visible(False)
        continue

    axes[idx].boxplot([num_cat0, num_cat1], labels=['No', 'Sí'])
    axes[idx].set_title(f"Distribución de {feature}")

    stat, p_value = stats.mannwhitneyu(num_cat0, num_cat1, alternative='two-sided')

    resultados.append({
        "variable": feature,
        "p_value": p_value
    })

    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
    axes[idx].set_xlabel(f"Mann-Whitney p={p_value:.4f} {sig}")

for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout(h_pad=2)
plt.show()

df_pvalues = pd.DataFrame(resultados)
df_pvalues

In [ ]:
import scipy
print(scipy.__version__)

#### para ordenar igual que en el exito para poner en el word, el mapeo

In [ ]:
# Orden deseado
orden = [
    'Grupo',
    'Localización',
    'ASA',
    'Derivación posterior',
    'Tipo anomalía',
    'Diabetes',
    'Anomalía anatómica',
    'Leucocitos Totales',
    'Fumador tabaco',
    'Cultivo',
    'Genero',
    'HTA',
    'Catéter JJ previo',
    'Lado',
    'Microorganismo',
    'Guy Score',
    'Puls',
    'Porcentaje Neutrófilos',
    'Dislipemia',
    'Localidad_cod',
    'Procalcitonina',
    'Tipo tratamiento',
    'Cultivo resultado',
    'Calibre vaina',
    'Dilatación ureteral'
]

# Renombrar las variables de tu dataframe
mapeo = {
    'GRUPO': 'Grupo',
    'LOCALIZACIÓN': 'Localización',
    'ASA': 'ASA',
    'TIPOANOMALIA': 'Tipo anomalía',
    'Diabetes': 'Diabetes',
    'ANOMALIAANATÓMICA': 'Anomalía anatómica',
    'LeucocitosTotales': 'Leucocitos Totales',
    'Fumador_tabaco': 'Fumador tabaco',
    'Cultresul': 'Cultivo resultado',
    'Genero': 'Genero',
    'HTA': 'HTA',
    'CATETERJJPREVIO': 'Catéter JJ previo',
    'lado': 'Lado',
    'Microorganismo': 'Microorganismo',
    'GUY_SCORE': 'Guy Score',
    'PULS': 'Puls',
    'ProcentajeNeutrofilos': 'Porcentaje Neutrófilos',
    'Dislipemia': 'Dislipemia',
    'LOCAL_COD': 'Localidad_cod',
    'Procalcitonina': 'Procalcitonina',
    'tipotto': 'Tipo tratamiento',
    'Calibre_vaina': 'Calibre vaina',
    'Dilatacion_ureteral': 'Dilatación ureteral'
}


In [ ]:
# =========================================================
# IMPORTS
# =========================================================
import math
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

# =========================================================
# CONFIG
# =========================================================
patient_data = patient_data.loc[:, ~patient_data.columns.duplicated()]

feat_cat1 = 'IngresoSiNo'

vars_separadas = ['TIPOANOMALIA']

results = []
excluded_vars = []

# =========================================================
# FUNCIÓN PVALOR
# =========================================================
def format_pvalue(p):
    if p < 0.0001:
        return "<0.0001"
    return f"{p:.4f}"

# =========================================================
# VALIDACIÓN (con diagnóstico de fallo)
# =========================================================
def variable_valida(df, yvar, xvar):

    reasons = []

    # -------------------------
    # 1. variables válidas
    # -------------------------
    if yvar == xvar:
        return False, ["misma variable"]

    if xvar not in df.columns:
        return False, ["no existe en el DataFrame"]

    # -------------------------
    # 2. subset limpio
    # -------------------------
    subset = df[[yvar, xvar]].dropna()

    if len(subset) == 0:
        return False, ["sin datos tras dropna"]

    # -------------------------
    # 3. cardinalidad
    # -------------------------
    x_unique = subset[xvar].nunique()
    y_unique = subset[yvar].nunique()

    if x_unique < 2:
        reasons.append("variable X con <2 categorías")

    if y_unique < 2:
        reasons.append("variable Y con <2 categorías")

    # -------------------------
    # 4. tabla contingencia
    # -------------------------
    tabla = pd.crosstab(subset[yvar], subset[xvar])

    if tabla.shape[0] < 2:
        reasons.append("tabla con <2 filas")

    if tabla.shape[1] < 2:
        reasons.append("tabla con <2 columnas")

    # -------------------------
    # salida segura
    # -------------------------
    if len(reasons) > 0:
        return False, reasons

    return True, []

# =========================================================
# HEATMAP
# =========================================================
def plot_contingencias(lista_variables):

    valid_vars = []

    for v in lista_variables:

        ok, reasons = variable_valida(patient_data, feat_cat1, v)

        if not ok:
            excluded_vars.append({
                "Variable": v,
                "Motivo": "; ".join(reasons)
            })
        else:
            valid_vars.append(v)

    if len(valid_vars) == 0:
        print("No hay variables válidas.")
        return

    n_cols = 2
    n_rows = math.ceil(len(valid_vars) / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 4))

    if n_rows == 1:
        axes = axes.reshape(1, -1)

    axes = axes.flatten()

    for i, var in enumerate(valid_vars):

        subset = patient_data[[feat_cat1, var]].dropna().copy()

        subset[feat_cat1] = subset[feat_cat1].map(
            mapeos.get(feat_cat1, lambda x: str(x))
        )

        subset[var] = subset[var].map(
            mapeos.get(var, lambda x: str(x))
        )

        raw_crosstab = pd.crosstab(subset[feat_cat1], subset[var])

        norm_crosstab = pd.crosstab(
            subset[feat_cat1],
            subset[var],
            normalize='index'
        ) * 100

        # =====================================================
        # FISHER (2x2 o RxC)
        # =====================================================
        tabla_np = raw_crosstab.to_numpy()

        try:
            resultado = fisher_exact(tabla_np)
            pvalue = resultado.pvalue

            if tabla_np.shape == (2, 2):
                metodo = "Fisher exacto"
            else:
                metodo = "Fisher-Freeman-Halton"

        except Exception as e:
            pvalue = float("nan")
            metodo = f"Error: {str(e)}"

        sns.heatmap(
            norm_crosstab,
            ax=axes[i],
            annot=raw_crosstab,
            fmt='d',
            cmap='Blues',
            linewidths=0.5,
            cbar=False,
            vmin=0,
            vmax=100
        )

        axes[i].set_title(
            f"{var}\n{metodo}: p={format_pvalue(pvalue) if pd.notnull(pvalue) else 'NA'}",
            fontsize=10
        )

        axes[i].set_xlabel('')
        axes[i].set_ylabel(feat_cat1)

        axes[i].tick_params(axis='x')

        results.append({
            'Variable': var,
            'Método': metodo,
            'p-valor': pvalue
        })

    for j in range(len(valid_vars), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

# =========================================================
# VARIABLES
# =========================================================
categoricas_main = [
    c for c in categoricas
    if c not in vars_separadas
    and c not in columnas_complicaciones
]

plot_contingencias(categoricas_main)
plot_contingencias(vars_separadas)

# =========================================================
# RESULTADOS
# =========================================================
results_df = pd.DataFrame(results)
excluded_df = pd.DataFrame(excluded_vars)

# OUTPUTS
excluded_df

In [ ]:
results_df

In [ ]:
# Copia del dataframe
df = results_df.copy()

# Renombrar
df['Variable'] = df['Variable'].replace(mapeo)

# Ordenar
df['Variable'] = pd.Categorical(
    df['Variable'],
    categories=orden,
    ordered=True
)

df = df.sort_values('Variable').reset_index(drop=True)

# Formatear p-valor con coma decimal
df['p-valor'] = (
    df['p-valor']
    .map(lambda x: f'{x:.4f}')
    .str.replace('.', ',', regex=False)
)

df

infeccion

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd

feat_cat = 'ComplInfSeps'

n_cols = 3
n_rows = -(-len(numericas) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()

resultados = []

for idx, feature in enumerate(numericas):
    subset = patient_data[[feat_cat, feature]].dropna()

    num_cat0 = subset.loc[subset[feat_cat] == 0, feature]
    num_cat1 = subset.loc[subset[feat_cat] == 1, feature]

    if num_cat0.empty or num_cat1.empty:
        axes[idx].set_visible(False)
        continue

    axes[idx].boxplot([num_cat0, num_cat1], labels=['No', 'Sí'])
    axes[idx].set_title(f"Distribución de {feature}")

    stat, p_value = stats.mannwhitneyu(num_cat0, num_cat1, alternative='two-sided')

    resultados.append({
        "variable": feature,
        "p_value": p_value
    })

    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
    axes[idx].set_xlabel(f"Mann-Whitney p={p_value:.4f} {sig}")

for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout(h_pad=2)
plt.show()

df_pvalues = pd.DataFrame(resultados)
df_pvalues

In [ ]:
# =========================================================
# IMPORTS
# =========================================================
import math
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

# =========================================================
# CONFIG
# =========================================================
patient_data = patient_data.loc[:, ~patient_data.columns.duplicated()]

feat_cat1 = 'ComplInfSeps'

vars_separadas = ['TIPOANOMALIA']

results = []
excluded_vars = []

# =========================================================
# FUNCIÓN PVALOR
# =========================================================
def format_pvalue(p):
    if p < 0.0001:
        return "<0.0001"
    return f"{p:.4f}"

# =========================================================
# VALIDACIÓN (con diagnóstico de fallo)
# =========================================================
def variable_valida(df, yvar, xvar):

    reasons = []

    # -------------------------
    # 1. variables válidas
    # -------------------------
    if yvar == xvar:
        return False, ["misma variable"]

    if xvar not in df.columns:
        return False, ["no existe en el DataFrame"]

    # -------------------------
    # 2. subset limpio
    # -------------------------
    subset = df[[yvar, xvar]].dropna()

    if len(subset) == 0:
        return False, ["sin datos tras dropna"]

    # -------------------------
    # 3. cardinalidad
    # -------------------------
    x_unique = subset[xvar].nunique()
    y_unique = subset[yvar].nunique()

    if x_unique < 2:
        reasons.append("variable X con <2 categorías")

    if y_unique < 2:
        reasons.append("variable Y con <2 categorías")

    # -------------------------
    # 4. tabla contingencia
    # -------------------------
    tabla = pd.crosstab(subset[yvar], subset[xvar])

    if tabla.shape[0] < 2:
        reasons.append("tabla con <2 filas")

    if tabla.shape[1] < 2:
        reasons.append("tabla con <2 columnas")

    # -------------------------
    # salida segura
    # -------------------------
    if len(reasons) > 0:
        return False, reasons

    return True, []

# =========================================================
# HEATMAP
# =========================================================
def plot_contingencias(lista_variables):

    valid_vars = []

    for v in lista_variables:

        ok, reasons = variable_valida(patient_data, feat_cat1, v)

        if not ok:
            excluded_vars.append({
                "Variable": v,
                "Motivo": "; ".join(reasons)
            })
        else:
            valid_vars.append(v)

    if len(valid_vars) == 0:
        print("No hay variables válidas.")
        return

    n_cols = 2
    n_rows = math.ceil(len(valid_vars) / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 4))

    if n_rows == 1:
        axes = axes.reshape(1, -1)

    axes = axes.flatten()

    for i, var in enumerate(valid_vars):

        subset = patient_data[[feat_cat1, var]].dropna().copy()

        subset[feat_cat1] = subset[feat_cat1].map(
            mapeos.get(feat_cat1, lambda x: str(x))
        )

        subset[var] = subset[var].map(
            mapeos.get(var, lambda x: str(x))
        )

        raw_crosstab = pd.crosstab(subset[feat_cat1], subset[var])

        norm_crosstab = pd.crosstab(
            subset[feat_cat1],
            subset[var],
            normalize='index'
        ) * 100

        # =====================================================
        # FISHER (2x2 o RxC)
        # =====================================================
        tabla_np = raw_crosstab.to_numpy()

        try:
            resultado = fisher_exact(tabla_np)
            pvalue = resultado.pvalue

            if tabla_np.shape == (2, 2):
                metodo = "Fisher exacto"
            else:
                metodo = "Fisher-Freeman-Halton"

        except Exception as e:
            pvalue = float("nan")
            metodo = f"Error: {str(e)}"

        sns.heatmap(
            norm_crosstab,
            ax=axes[i],
            annot=raw_crosstab,
            fmt='d',
            cmap='Blues',
            linewidths=0.5,
            cbar=False,
            vmin=0,
            vmax=100
        )

        axes[i].set_title(
            f"{var}\n{metodo}: p={format_pvalue(pvalue) if pd.notnull(pvalue) else 'NA'}",
            fontsize=10
        )

        axes[i].set_xlabel('')
        axes[i].set_ylabel(feat_cat1)

        axes[i].tick_params(axis='x')

        results.append({
            'Variable': var,
            'Método': metodo,
            'p-valor': pvalue
        })

    for j in range(len(valid_vars), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

# =========================================================
# VARIABLES
# =========================================================
categoricas_main = [
    c for c in categoricas
    if c not in vars_separadas
    and c not in columnas_complicaciones and c not in ['RESUELTOEN6M','IngresoSiNo']
]

plot_contingencias(categoricas_main)
plot_contingencias(vars_separadas)

# =========================================================
# RESULTADOS
# =========================================================
results_df = pd.DataFrame(results)
excluded_df = pd.DataFrame(excluded_vars)

# OUTPUTS
excluded_df

In [ ]:
results_df

In [ ]:
# Copia del dataframe
df = results_df.copy()

# Renombrar
df['Variable'] = df['Variable'].replace(mapeo)

# Ordenar
df['Variable'] = pd.Categorical(
    df['Variable'],
    categories=orden,
    ordered=True
)

df = df.sort_values('Variable').reset_index(drop=True)

# Formatear p-valor con coma decimal
df['p-valor'] = (
    df['p-valor']
    .map(lambda x: f'{x:.4f}')
    .str.replace('.', ',', regex=False)
)

df

sangrados

In [ ]:
patient_data["ComplHemo"].value_counts()

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd

feat_cat = 'ComplHemo'

n_cols = 3
n_rows = -(-len(numericas) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()

resultados = []

for idx, feature in enumerate(numericas):
    subset = patient_data[[feat_cat, feature]].dropna()

    num_cat0 = subset.loc[subset[feat_cat] == 0, feature]
    num_cat1 = subset.loc[subset[feat_cat] == 1, feature]

    if num_cat0.empty or num_cat1.empty:
        axes[idx].set_visible(False)
        continue

    axes[idx].boxplot([num_cat0, num_cat1], labels=['No', 'Sí'])
    axes[idx].set_title(f"Distribución de {feature}")

    stat, p_value = stats.mannwhitneyu(num_cat0, num_cat1, alternative='two-sided')

    resultados.append({
        "variable": feature,
        "p_value": p_value
    })

    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
    axes[idx].set_xlabel(f"Mann-Whitney p={p_value:.4f} {sig}")

for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout(h_pad=2)
plt.show()

df_pvalues = pd.DataFrame(resultados)
df_pvalues

In [ ]:
# =========================================================
# IMPORTS
# =========================================================
import math
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

# =========================================================
# CONFIG
# =========================================================
patient_data = patient_data.loc[:, ~patient_data.columns.duplicated()]

feat_cat1 = 'ComplHemo'

vars_separadas = ['TIPOANOMALIA']

results = []
excluded_vars = []

# =========================================================
# FUNCIÓN PVALOR
# =========================================================
def format_pvalue(p):
    if p < 0.0001:
        return "<0.0001"
    return f"{p:.4f}"

# =========================================================
# VALIDACIÓN (con diagnóstico de fallo)
# =========================================================
def variable_valida(df, yvar, xvar):

    reasons = []

    # -------------------------
    # 1. variables válidas
    # -------------------------
    if yvar == xvar:
        return False, ["misma variable"]

    if xvar not in df.columns:
        return False, ["no existe en el DataFrame"]

    # -------------------------
    # 2. subset limpio
    # -------------------------
    subset = df[[yvar, xvar]].dropna()

    if len(subset) == 0:
        return False, ["sin datos tras dropna"]

    # -------------------------
    # 3. cardinalidad
    # -------------------------
    x_unique = subset[xvar].nunique()
    y_unique = subset[yvar].nunique()

    if x_unique < 2:
        reasons.append("variable X con <2 categorías")

    if y_unique < 2:
        reasons.append("variable Y con <2 categorías")

    # -------------------------
    # 4. tabla contingencia
    # -------------------------
    tabla = pd.crosstab(subset[yvar], subset[xvar])

    if tabla.shape[0] < 2:
        reasons.append("tabla con <2 filas")

    if tabla.shape[1] < 2:
        reasons.append("tabla con <2 columnas")

    # -------------------------
    # salida segura
    # -------------------------
    if len(reasons) > 0:
        return False, reasons

    return True, []

# =========================================================
# HEATMAP
# =========================================================
def plot_contingencias(lista_variables):

    valid_vars = []

    for v in lista_variables:

        ok, reasons = variable_valida(patient_data, feat_cat1, v)

        if not ok:
            excluded_vars.append({
                "Variable": v,
                "Motivo": "; ".join(reasons)
            })
        else:
            valid_vars.append(v)

    if len(valid_vars) == 0:
        print("")
        return

    n_cols = 2
    n_rows = math.ceil(len(valid_vars) / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 4))

    if n_rows == 1:
        axes = axes.reshape(1, -1)

    axes = axes.flatten()

    for i, var in enumerate(valid_vars):

        subset = patient_data[[feat_cat1, var]].dropna().copy()

        subset[feat_cat1] = subset[feat_cat1].map(
            mapeos.get(feat_cat1, lambda x: str(x))
        )

        subset[var] = subset[var].map(
            mapeos.get(var, lambda x: str(x))
        )

        raw_crosstab = pd.crosstab(subset[feat_cat1], subset[var])

        norm_crosstab = pd.crosstab(
            subset[feat_cat1],
            subset[var],
            normalize='index'
        ) * 100

        # =====================================================
        # FISHER (2x2 o RxC)
        # =====================================================
        tabla_np = raw_crosstab.to_numpy()

        try:
            resultado = fisher_exact(tabla_np)
            pvalue = resultado.pvalue

            if tabla_np.shape == (2, 2):
                metodo = "Fisher exacto"
            else:
                metodo = "Fisher-Freeman-Halton"

        except Exception as e:
            pvalue = float("nan")
            metodo = f"Error: {str(e)}"

        sns.heatmap(
            norm_crosstab,
            ax=axes[i],
            annot=raw_crosstab,
            fmt='d',
            cmap='Blues',
            linewidths=0.5,
            cbar=False,
            vmin=0,
            vmax=100
        )

        axes[i].set_title(
            f"{var}\n{metodo}: p={format_pvalue(pvalue) if pd.notnull(pvalue) else 'NA'}",
            fontsize=10
        )

        axes[i].set_xlabel('')
        axes[i].set_ylabel(feat_cat1)

        axes[i].tick_params(axis='x')

        results.append({
            'Variable': var,
            'Método': metodo,
            'p-valor': pvalue
        })

    for j in range(len(valid_vars), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

# =========================================================
# VARIABLES
# =========================================================
categoricas_main = [
    c for c in categoricas
    if c not in vars_separadas
    and c not in columnas_complicaciones and c not in ['RESUELTOEN6m','IngresoSiNo']
]

plot_contingencias(categoricas_main)
plot_contingencias(vars_separadas)

# =========================================================
# RESULTADOS
# =========================================================
results_df = pd.DataFrame(results)
excluded_df = pd.DataFrame(excluded_vars)

# OUTPUTS
excluded_df

In [ ]:
results_df

In [ ]:
# Copia del dataframe
df = results_df.copy()

# Renombrar
df['Variable'] = df['Variable'].replace(mapeo)

# Ordenar
df['Variable'] = pd.Categorical(
    df['Variable'],
    categories=orden,
    ordered=True
)

df = df.sort_values('Variable').reset_index(drop=True)

# Formatear p-valor con coma decimal
df['p-valor'] = (
    df['p-valor']
    .map(lambda x: f'{x:.4f}')
    .str.replace('.', ',', regex=False)
)

df

Variable anestesia no tiene varianza y todos los valores son "1" ('Vaina_acceso'), por lo cual se elimina del estudio

In [ ]:
# Verificar si la columna tiene solo NaN
patient_data['Vaina_acceso'].isna().sum()

# Verificar si todos los valores son iguales
patient_data['Vaina_acceso'].nunique()
patient_data = patient_data.drop(columns=['Vaina_acceso'])

categoricas.remove('Vaina_acceso')

In [ ]:
patient_data_complic = patient_data.copy()

In [ ]:
patient_data_complic

In [ ]:
patient_data_complic.drop(columns=['RESUELTOEN6m'], inplace=True)

# Complicaciones + Ingreso

## tratar nasns

In [ ]:
patient_data_complic.dropna(subset=['ComplInfSeps'],inplace=True)
patient_data_complic


In [ ]:
patient_data_complic.isna().sum()

In [ ]:
# Contar los NaNs por fila
nans_por_fila = patient_data_complic.isna().sum(axis=1)
# Filtrar solo las filas que tienen más de 0 NaNs
nans_por_fila_con_na = nans_por_fila[nans_por_fila > 0]
#print(nans_por_fila_con_na)
print('Total de elementos con al menos un NaN:',len(nans_por_fila_con_na)) 
cantidad_por_nans = nans_por_fila.value_counts().sort_index()
print('La cantidad de Nans en una fila y el número de filas con esta cantidad:')
print(cantidad_por_nans)

In [ ]:
data_nan = np.isnan(patient_data_complic.drop(columns=['codigo']))
nans_per_colum = np.sum(data_nan, axis=0)
plt.xticks(rotation=90)
plt.bar(patient_data_complic.drop(columns=['codigo']).columns, nans_per_colum)
plt.show()

In [ ]:
patient_data_complic["Fumador_tabaco"] = patient_data_complic["Fumador_tabaco"].fillna(0).astype(int)

In [ ]:
## threshold = 0.3

## na_ratio = patient_data_complic.isnull().mean()

## cols_to_keep = na_ratio[na_ratio < threshold].index.tolist()

## # asegurar que TIPOANOMALIA siempre esté
## if "TIPOANOMALIA" not in cols_to_keep:
##     cols_to_keep.append("TIPOANOMALIA")

## patient_data_complic = patient_data_complic[cols_to_keep]

In [ ]:
imputar = np.round(patient_data[numericas].median())  # patient_data_complic
imputar

In [ ]:
### categorical_cols = ['tipotto','LOCAL_COD','GRUPO','GUY_SCORE','Procalcitonina']
### patient_data_complic[categorical_cols].mode()

In [ ]:
patient_data_complic[numericas] = patient_data_complic[numericas].fillna(imputar)
### patient_data_complic[categorical_cols] = patient_data_complic[categorical_cols].fillna(patient_data_complic[categorical_cols].mode().iloc[0])
patient_data_complic.head(50)

In [ ]:
data_nan = np.isnan(patient_data_complic.drop(columns=['codigo']))
nans_per_colum = np.sum(data_nan, axis=0)
plt.xticks(rotation=90)
plt.bar(patient_data_complic.drop(columns=['codigo']).columns, nans_per_colum)
plt.show()

In [ ]:
patient_data_complic.columns

In [ ]:
categorias = ['Genero', 'lado', 'cultivo', 'Cultresul', 'Microorganismo','Diabetes', 'HTA', 'Dislipemia', 'Fumador_tabaco', 'ASA', 'ANOMALIAANATÓMICA', 'TIPOANOMALIA',   'GRUPO',  'CATETERJJPREVIO',   'DERIVACIONPOSTERIOR',  'LeucocitosTotales', 'ProcentajeNeutrofilos','LOCALIZACIÓN', 'Procalcitonina','tipotto','LOCAL_COD','GUY_SCORE'] # ,'Dilatacion_ureteral','Calibre_vaina','PULS', '
patient_data_complic[categorias] = patient_data_complic[categorias].astype('Int64')
patient_data_complic.head()

In [ ]:
patient_data_complic["Diabetes"] = patient_data_complic["Diabetes"].replace({1: 2, 2: 1})

In [ ]:
cat_nominales = [ 'lado','cultivo', 'Cultresul', 'Microorganismo','Fumador_tabaco', 'TIPOANOMALIA', 'DERIVACIONPOSTERIOR'] #,
patient_data_complic = pd.get_dummies(
    patient_data_complic,
    columns=cat_nominales,
    dtype=int
)
categorical_cols_con_na = ['tipotto','LOCAL_COD','GRUPO','GUY_SCORE','Procalcitonina', 'LOCALIZACIÓN']
patient_data_complic = pd.get_dummies(
    patient_data_complic, dummy_na=True,
    columns=categorical_cols_con_na,
    dtype=int
)

In [ ]:
patient_data_complic.head()

In [ ]:
data_nan = np.isnan(patient_data_complic.drop(columns=['codigo']))
nans_per_colum = np.sum(data_nan, axis=0)
plt.xticks(rotation=90)
plt.bar(patient_data_complic.drop(columns=['codigo']).columns, nans_per_colum)
plt.show()

In [ ]:
patient_data_complic.columns

In [ ]:
(patient_data_complic["cultivo_1"]
 .value_counts(normalize=True)
 .mul(100)
 .round(2))

In [ ]:
(patient_data_complic["DERIVACIONPOSTERIOR_2"]
 .value_counts(normalize=True)
 .mul(100)
 .round(2))

In [ ]:
patient_data_complic.drop(columns=['cultivo_1','DERIVACIONPOSTERIOR_2'], inplace=True)

INFECCIOSAS

In [ ]:
# Contar cuántas veces aparece cada ID de paciente en la columna 'n_historia'
repeated_patients = patient_data_complic['codigo'].value_counts()
# Filtrar solo aquellos que se repiten más de una vez (mayor a 1)
repeated_patients = repeated_patients[repeated_patients > 1]
# Contar la frecuencia de cada número de repeticiones
repetition_counts = repeated_patients.value_counts().sort_index()
# Mostrar cuántos pacientes hay con 2 repeticiones, 3 repeticiones, etc.
print(repetition_counts)
# Mostrar la suma total de pacientes repetidos
print("Total de pacientes repetidos:", repetition_counts.sum())

In [ ]:
patient_data_complic_inf = patient_data_complic.copy()
patient_data_complic_inf.drop(columns=['ComplHemo','IngresoSiNo'], inplace=True)

SANGRADOS

In [ ]:
patient_data_complic_hemo = patient_data_complic.copy()
patient_data_complic.dropna(subset=['ComplHemo'],inplace=True)
patient_data_complic_hemo.drop(columns=['ComplInfSeps','IngresoSiNo'], inplace=True)

INGRESO

In [ ]:
patient_data_complic_ingreso = patient_data_complic.copy()
patient_data_complic_ingreso.drop(columns=['ComplInfSeps','ComplHemo'], inplace=True)

In [ ]:
patient_data_complic_ingreso

# ML

### Infecciones

In [ ]:
results_inf = []
results_train = []

In [ ]:
X_train = patient_data_complic_inf.drop(columns=["ComplInfSeps",'codigo'])
y_train = patient_data_complic_inf["ComplInfSeps"]

In [ ]:
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    make_scorer,
    cohen_kappa_score,
    matthews_corrcoef,
    confusion_matrix,
    recall_score,
    precision_score
)

import numpy as np
import pandas as pd


def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) != 0 else 0


def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn) if (tn + fn) != 0 else 0


def grid_search_cv_model(
    model,
    param_grid,
    X_train,
    y_train,
    refit_metric="auc",
    scale_data=False,
    n_splits=5,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
    verbose=1
):

    scoring = {
        "auc": "roc_auc",
        "accuracy": "accuracy",
        "sensitivity": make_scorer(recall_score),
        "specificity": make_scorer(specificity_score),
        "ppv": make_scorer(precision_score),
        "npv": make_scorer(npv_score),
        "kappa": make_scorer(cohen_kappa_score),
        "mcc": make_scorer(matthews_corrcoef)
    }

    estimator = Pipeline([
        ("scaler", MinMaxScaler()),
        ("model", model)
    ]) if scale_data else model

    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=random_state
    )

    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=scoring,
        refit=refit_metric,
        cv=cv,
        n_jobs=n_jobs,
        verbose=verbose,
        return_train_score=True
    )

    search.fit(X_train, y_train)

    best_idx = search.best_index_

    # =========================
    # TEST CV (mean/std)
    # =========================
    def get(metric):
        return (
            search.cv_results_[f"mean_test_{metric}"][best_idx],
            search.cv_results_[f"std_test_{metric}"][best_idx],
            search.cv_results_[f"mean_train_{metric}"][best_idx],
            search.cv_results_[f"std_train_{metric}"][best_idx],
        )

    auc_mean, auc_std, auc_tr_m, auc_tr_s = get("auc")
    acc_mean, acc_std, acc_tr_m, acc_tr_s = get("accuracy")
    sens_mean, sens_std, sens_tr_m, sens_tr_s = get("sensitivity")
    spec_mean, spec_std, spec_tr_m, spec_tr_s = get("specificity")
    ppv_mean, ppv_std, ppv_tr_m, ppv_tr_s = get("ppv")
    npv_mean, npv_std, npv_tr_m, npv_tr_s = get("npv")
    kappa_mean, kappa_std, kappa_tr_m, kappa_tr_s = get("kappa")
    mcc_mean, mcc_std, mcc_tr_m, mcc_tr_s = get("mcc")

    metrics_df = pd.DataFrame({
        "metric": [
            "AUC", "Accuracy", "Sensitivity", "Specificity",
            "PPV", "NPV", "Kappa", "MCC"
        ],
        "train_mean": [
            auc_tr_m, acc_tr_m, sens_tr_m, spec_tr_m,
            ppv_tr_m, npv_tr_m, kappa_tr_m, mcc_tr_m
        ],
        "train_std": [
            auc_tr_s, acc_tr_s, sens_tr_s, spec_tr_s,
            ppv_tr_s, npv_tr_s, kappa_tr_s, mcc_tr_s
        ],
        "test_mean": [
            auc_mean, acc_mean, sens_mean, spec_mean,
            ppv_mean, npv_mean, kappa_mean, mcc_mean
        ],
        "test_std": [
            auc_std, acc_std, sens_std, spec_std,
            ppv_std, npv_std, kappa_std, mcc_std
        ]
    })

    print("\n=========================")
    print("RESULTADOS CV")
    print("=========================\n")
    print(metrics_df.round(3))

    print("\n=========================")
    print("MEJORES HIPERPARÁMETROS")
    print("=========================\n")
    print(search.best_params_)

    best_model = search.best_estimator_

    summary = {
        "model": type(model).__name__,
        "refit_metric": refit_metric,
        "auc_mean": auc_mean,
        "auc_std": auc_std,
        "accuracy_mean": acc_mean,
        "accuracy_std": acc_std,
        "kappa_mean": kappa_mean,
        "kappa_std": kappa_std,
        "mcc_mean": mcc_mean,
        "mcc_std": mcc_std,
        "best_params": search.best_params_
    }

    return best_model, search, summary, metrics_df

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

#  =====================================================
# MODELO
# =====================================================

logreg_model = LogisticRegression(
    random_state=42,
    max_iter=5000
)

# =====================================================
# HIPERPARÁMETROS
# =====================================================

param_grid = {
    "model__C": [0.01, 0.1, 1, 10],

    "model__penalty": ["l1", "l2"],

    "model__solver": ["liblinear"],
    "model__class_weight": [
        None,
        "balanced",
        {0: 3, 1: 0},
        {0: 5, 1: 0}
    ]
}

best_model_logreg_auc, search, summary_logreg_auc, metrics_df_logreg_auc = grid_search_cv_model(
    model=logreg_model,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    refit_metric="auc",
    scale_data=True,
    random_state=42,
    n_jobs=-1
)

results_inf.append(summary_logreg_auc)
results_train.append(metrics_df_logreg_auc)

In [ ]:
best_model_logreg_kappa, search, summary_logreg_kappa, metrics_df_logreg_kappa = grid_search_cv_model(
    model=logreg_model,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    refit_metric="kappa",
    scale_data=True,
    random_state=42,
    n_jobs=-1
)

results_inf.append(summary_logreg_kappa)
results_train.append(metrics_df_logreg_kappa)

In [ ]:
best_model_logreg_mcc, search, summary_logreg_mcc, metrics_df_logreg_mcc = grid_search_cv_model(
    model=logreg_model,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    refit_metric="mcc",
    scale_data=True,
    random_state=42,
    n_jobs=-1
)

results_inf.append(summary_logreg_mcc)
results_train.append(metrics_df_logreg_mcc)

In [ ]:
results_train

In [ ]:
import pandas as pd

nombres_filas = ['AUC', 'Kappa', 'MCC']
filas = []

for nombre, df in zip(nombres_filas, results_train):

    fila = {'Modelo': nombre}

    for _, row in df.iterrows():
        valor = (
            f"{row['test_mean']:.3f}".replace('.', ',')
            + '±' +
            f"{row['test_std']:.3f}".replace('.', ',')
        )
        fila[row['metric']] = valor

    filas.append(fila)

df_final = pd.DataFrame(filas).set_index('Modelo')

df_final

In [ ]:
df_final.T

In [ ]:
def traducir_variable(var, mapeos):
    # Variables que no vienen de one-hot
    if "_" not in var:
        return var

    partes = var.split("_")

    # Buscar el nombre de la variable más largo posible
    for i in range(len(partes)-1, 0, -1):
        nombre = "_".join(partes[:i])
        valor = "_".join(partes[i:])

        if nombre in mapeos:
            # categoría NaN
            if valor == "<NA>":
                return f"{nombre}: Desconocido"

            try:
                valor = int(valor)
                etiqueta = mapeos[nombre].get(valor, str(valor))
                return f"{nombre}: {etiqueta}"
            except:
                return var

    return var

In [ ]:
logreg = best_model_logreg_auc.named_steps["model"]

coef_df = pd.DataFrame({
    "variable": X_train.columns,
    "coeficiente": logreg.coef_[0]
})

coef_df["odds_ratio"] = np.exp(coef_df["coeficiente"])
coef_df["importancia_abs"] = np.abs(coef_df["coeficiente"])

coef_df = coef_df.sort_values(
    by="importancia_abs",
    ascending=False
)

print(coef_df)
coef_df.head(20)

coef_df["variable_bonita"] = (
    coef_df["variable"]
    .apply(lambda x: traducir_variable(x, mapeos))
)

In [ ]:
import matplotlib.pyplot as plt

top_n = 20

plot_df = (
    coef_df
    .head(top_n)
    .sort_values(by="coeficiente")
)

plt.figure(figsize=(10,8))

plt.barh(
    plot_df["variable_bonita"],
    plot_df["coeficiente"]
)

plt.axvline(
    x=0,
    color="black",
    linestyle="--"
)

plt.title("Coeficientes de la regresión logística")
plt.xlabel("Coeficiente")
plt.ylabel("")

plt.tight_layout()
plt.show()

ingreso

In [ ]:
X_train = patient_data_complic_ingreso.drop(columns=["IngresoSiNo",'codigo'])
y_train = patient_data_complic_ingreso["IngresoSiNo"]
results_ing=[]
results_train_ing=[]

In [ ]:
best_model_logreg_auc_ing, search, summary_logreg_auc_ing, metrics_df_logreg_auc_ing = grid_search_cv_model(
    model=logreg_model,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    refit_metric="auc",
    scale_data=True,
    random_state=42,
    n_jobs=-1
)

results_ing.append(summary_logreg_auc_ing)
results_train_ing.append(metrics_df_logreg_auc_ing)

In [ ]:
best_model_logreg_kappa_ing, search, summary_logreg_kappa_ing, metrics_df_logreg_kappa_ing = grid_search_cv_model(
    model=logreg_model,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    refit_metric="kappa",
    scale_data=True,
    random_state=42,
    n_jobs=-1
)

results.append(summary_logreg_kappa_ing)
results_train_ing.append(metrics_df_logreg_kappa_ing)

In [ ]:
best_model_logreg_mcc_ing, search, summary_logreg_mcc_ing, metrics_df_logreg_mcc_ing = grid_search_cv_model(
    model=logreg_model,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    refit_metric="mcc",
    scale_data=True,
    random_state=42,
    n_jobs=-1
)

results.append(summary_logreg_mcc_ing)
results_train_ing.append(metrics_df_logreg_mcc_ing)

In [ ]:
import pandas as pd

nombres_filas = ['AUC', 'Kappa', 'MCC']
filas = []

for nombre, df in zip(nombres_filas, results_train_ing):

    fila = {'Modelo': nombre}

    for _, row in df.iterrows():
        valor = (
            f"{row['test_mean']:.3f}".replace('.', ',')
            + '±' +
            f"{row['test_std']:.3f}".replace('.', ',')
        )
        fila[row['metric']] = valor

    filas.append(fila)

df_final_ing = pd.DataFrame(filas).set_index('Modelo')

df_final_ing.T

In [ ]:
logreg = best_model_logreg_auc_ing.named_steps["model"]

coef_df = pd.DataFrame({
    "variable": X_train.columns,
    "coeficiente": logreg.coef_[0]
})

coef_df["odds_ratio"] = np.exp(coef_df["coeficiente"])
coef_df["importancia_abs"] = np.abs(coef_df["coeficiente"])

coef_df = coef_df.sort_values(
    by="importancia_abs",
    ascending=False
)

print(coef_df)
coef_df.head(20)

coef_df["variable_bonita"] = (
    coef_df["variable"]
    .apply(lambda x: traducir_variable(x, mapeos))
)

In [ ]:
import matplotlib.pyplot as plt
top_n = 20

plot_df = (
    coef_df
    .head(top_n)
    .sort_values(by="coeficiente")
)

plt.figure(figsize=(10,8))

plt.barh(
    plot_df["variable_bonita"],
    plot_df["coeficiente"]
)

plt.axvline(
    x=0,
    color="black",
    linestyle="--"
)

plt.title("Coeficientes de la regresión logística")
plt.xlabel("Coeficiente")
plt.ylabel("")

plt.tight_layout()
plt.show()

hemo

In [ ]:
X_train = patient_data_complic_hemo.drop(columns=["ComplHemo",'codigo'])
y_train = patient_data_complic_hemo["ComplHemo"]
results_hemo=[]
results_train_hemo=[]

In [ ]:
patient_data_complic_hemo = patient_data_complic_hemo.dropna(
    subset=["ComplHemo"]
)

X_train = patient_data_complic_hemo.drop(
    columns=["ComplHemo", "codigo"]
)

y_train = patient_data_complic_hemo["ComplHemo"]

In [ ]:
y_train.isna().sum()

In [ ]:
patient_data_complic_hemo["ComplHemo"].isna().sum()

In [ ]:
best_model_logreg_auc_hemo, search, summary_logreg_auc_hemo, metrics_df_logreg_auc_hemo = grid_search_cv_model(
    model=logreg_model,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    refit_metric="auc",
    scale_data=True,
    n_splits=2,
    random_state=42,
    n_jobs=-1
)

results_ing.append(summary_logreg_auc_hemo)
results_train_hemo.append(metrics_df_logreg_auc_hemo)

In [ ]:
best_model_logreg_kappa_hemo, search, summary_logreg_kappa_hemo, metrics_df_logreg_kappa_hemo = grid_search_cv_model(
    model=logreg_model,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    refit_metric="kappa",
    scale_data=True,
    n_splits=2,
    random_state=42,
    n_jobs=-1
)

results.append(summary_logreg_kappa_hemo)
results_train_hemo.append(metrics_df_logreg_kappa_hemo)

In [ ]:
best_model_logreg_mcc_hemo, search, summary_logreg_mcc_hemo, metrics_df_logreg_mcc_hemo = grid_search_cv_model(
    model=logreg_model,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    refit_metric="mcc",
    scale_data=True,
    n_splits=2,
    random_state=42,
    n_jobs=-1
)

results.append(summary_logreg_mcc_hemo)
results_train_hemo.append(metrics_df_logreg_mcc_hemo)

In [ ]:
import pandas as pd

nombres_filas = ['AUC', 'Kappa', 'MCC']
filas = []

for nombre, df in zip(nombres_filas, results_train_hemo):

    fila = {'Modelo': nombre}

    for _, row in df.iterrows():
        valor = (
            f"{row['test_mean']:.3f}".replace('.', ',')
            + '±' +
            f"{row['test_std']:.3f}".replace('.', ',')
        )
        fila[row['metric']] = valor

    filas.append(fila)

df_final_hemo = pd.DataFrame(filas).set_index('Modelo')

df_final_hemo.T

In [ ]:
logreg = best_model_logreg_mcc_hemo.named_steps["model"]

coef_df = pd.DataFrame({
    "variable": X_train.columns,
    "coeficiente": logreg.coef_[0]
})

coef_df["odds_ratio"] = np.exp(coef_df["coeficiente"])
coef_df["importancia_abs"] = np.abs(coef_df["coeficiente"])

coef_df = coef_df.sort_values(
    by="importancia_abs",
    ascending=False
)

print(coef_df)
coef_df.head(20)

coef_df["variable_bonita"] = (
    coef_df["variable"]
    .apply(lambda x: traducir_variable(x, mapeos))
)

In [ ]:
coef_df.head(20)

In [ ]:
import matplotlib.pyplot as plt

top_n = 20

plot_df = (
    coef_df
    .head(top_n)
    .sort_values(by="coeficiente")
)

plt.figure(figsize=(10,8))

plt.barh(
    plot_df["variable_bonita"],
    plot_df["coeficiente"]
)

plt.axvline(
    x=0,
    color="black",
    linestyle="--"
)

plt.title("Coeficientes de la regresión logística")
plt.xlabel("Coeficiente")
plt.ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
patient_data_complic.iloc[:, 10:25]